# 02. Exploratory Data Analysis: ARTA Smell Dataset

Analyzes the authentic ARTA dataset (Zakeri-Nasrabadi 2020, Zenodo record 4266727).
Inspects the 4 spreadsheet datasets (DS1 to DS4) and smell distributions.


In [1]:
from pathlib import Path

import pandas as pd

data_dir = Path('../data/raw/arta/extracted/ARTADataset_release1/Datasets')
files = sorted(data_dir.glob('DS*.xlsx'))
dfs = {f.stem: pd.read_excel(f) for f in files}
for name, d in dfs.items():
    print(f'{name}: shape={d.shape}')


C:\Users\2005v\OneDrive\Desktop\CheckSense\.venv\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


DS1: shape=(985, 12)
DS2: shape=(1092, 10)
DS3: shape=(1522, 10)
DS4: shape=(1153, 10)


C:\Users\2005v\OneDrive\Desktop\CheckSense\.venv\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


## 2. Smell Columns in DS1


In [2]:
ds1 = dfs['DS1']
print('Columns in DS1:', list(ds1.columns))
smell_cols = [c for c in ds1.columns if c not in ('Requirement_text', 'File')]
smell_freq = {}
for col in smell_cols:
    present = ds1[col].astype(str).str.strip().ne('-') & ds1[col].notna()
    smell_freq[col] = present.sum()
pd.DataFrame(list(smell_freq.items()), columns=['Smell Type', 'Count in DS1']).sort_values('Count in DS1', ascending=False)


Columns in DS1: ['Requirement_text', 'File', 'Subjective_lang.', 'Ambiguous_adv._adj.', 'Loophole', 'Nonverifiable_term', 'Superlative', 'Comparative', 'Negative', 'Vague_pron.', 'Uncertain_verb', 'Polysemy']


,Smell Type,Count in DS1
9,Polysemy,436
0,Subjective_lang.,259
8,Uncertain_verb,188
3,Nonverifiable_term,165
7,Vague_pron.,73
1,Ambiguous_adv._adj.,69
6,Negative,42
5,Comparative,26
4,Superlative,8
2,Loophole,0


## 3. Sample 20 Rows from DS1 (Seed 42) - Metadata Only


In [3]:
sample_20 = ds1.sample(n=20, random_state=42)
records = []
for i, row in sample_20.reset_index(drop=True).iterrows():
    active_smells = [c for c in smell_cols if str(row[c]).strip() not in ('-', 'nan')]
    records.append({
        'Sample': i + 1,
        'File': row['File'],
        'Char_Length': len(str(row['Requirement_text'])),
        'Active_Smells_Count': len(active_smells),
        'Smell_Types': ', '.join(active_smells) if active_smells else 'None'
    })
pd.DataFrame(records)


,Sample,File,Char_Length,Active_Smells_Count,Smell_Types
0,1,2007-eirene_fun_7-2.xml,208,2,"Subjective_lang., Polysemy"
1,2,2007-eirene_fun_7-2.xml,160,2,"Subjective_lang., Uncertain_verb"
2,3,2007-eirene_fun_7-2.xml,349,2,"Superlative, Polysemy"
3,4,2007-eirene_fun_7-2.xml,209,1,Polysemy
4,5,0000 - cctns.xml,210,2,"Ambiguous_adv._adj., Polysemy"
5,6,2007-eirene_fun_7-2.xml,96,0,None
6,7,2007-eirene_fun_7-2.xml,287,3,"Negative, Uncertain_verb, Polysemy"
7,8,0000 - cctns.xml,218,1,Uncertain_verb
8,9,2007-eirene_fun_7-2.xml,111,2,"Ambiguous_adv._adj., Polysemy"
9,10,2007-eirene_fun_7-2.xml,119,1,Polysemy
